In [5]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
    
from sklearn.linear_model import LinearRegression
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

from src.network_simulator import MultiplexSimulator, create_orthogonal_sets
from src.variational_bayes import VariationalBayes
from src.helper_functions.phi_gradients import compute_gradient_theta_phi_k

In [2]:
warnings.filterwarnings("ignore", category=FutureWarning, 
                        message=".*force_all_finite.*")

# Simulate from the model

This is an example of simulating from the model with $N=100$ nodes and $L=20$ layers. We have 3 global and 3 layer groups, with feature vectors sampled from a multivariate normal distribution on $\mathbb{R}^3$. The nodes are split across the global groups as $(40, 40, 20)$. The probability of layer group assignment given global group are vectors of the form $(1 - 2\alpha, \alpha, \alpha)$. A function is included to randomly sample the layer level groups using these probability vectors, but this could be coded deterministically. Furthermore, the argument \texttt{num_adj_samps} specifies how many (independent) adjacency matrices you want to sample.

In [3]:
num_nodes = 100
num_layers = 20
num_global_groups = 3
max_num_layer_groups = 3
features_length = 3
num_adj_samps = 1

# Create features from multivariate normal distributions
global_group_sizes = [40, 40, 20]
features = np.zeros((num_nodes, features_length))
means = [
    [5, 5, 5],
    [0, 0, 0], # Mean for group 1
    [-5, -5, -5]]
covariances = [
    [[1, 0, 0], [0, 1, 0], [0, 0, 1]],   # Covariance for group 1
    [[1, 0, 0], [0, 1, 0], [0, 0, 1]], # Covariance for group 2
    [[1, 0, 0], [0, 1, 0], [0, 0, 1]]    # Covariance for group 3
]
# Generate features for each group
start_idx = 0
for i, (mean, cov) in enumerate(zip(means, covariances)):
    end_idx = start_idx + global_group_sizes[i]
    features[start_idx:end_idx, :features_length] = (
        np.random.multivariate_normal(mean, cov, size=global_group_sizes[i])
        )
    start_idx = end_idx

# 0 mean, 1 variance scaling
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Connectivity matrix
rho_matrix = np.array(
    [[0.9, 0.5, 0.2],
     [0.4, 0.7, 0.05],
     [0.2, 0.01, 0.6]]
)

# Layer probability vectors
alpha = 0.15
gamma1 = [1-2*alpha, alpha, alpha]
gamma2 = [alpha, 1-2*alpha, alpha]
gamma3 = [alpha, alpha, 1-2*alpha]

# A function to sample the layer groups.
def sample_layer_groups(num_layers, gammas, global_group_sizes):
    num_layer_groups = len(gammas[0])
    num_glob_groups = len(global_group_sizes)
    N = np.array(global_group_sizes).sum()
    layer_groups = np.zeros((num_layers, num_nodes))

    for layer in range(num_layers):
        samples = []
        for global_group in range(num_glob_groups):
            samples.append(
                np.random.choice(num_layer_groups,
                                 p=gammas[global_group],
                                 size=global_group_sizes[global_group])
            )
        layer_groups[layer,:] = (
            np.concatenate(samples)
        )

    return layer_groups

# Sample the layer groups
layer_groups = sample_layer_groups(num_layers, 
                                   [gamma1, gamma2, gamma3], 
                                   global_group_sizes)
layer_groups = layer_groups.astype(int)

# Simulate from the model
MS = MultiplexSimulator(num_nodes=num_nodes, num_layers=num_layers, 
                        num_glob_groups=num_global_groups,
                        max_num_layer_groups=max_num_layer_groups,
                        features=features, specify=True, 
                        rho_matrix=rho_matrix, layer_groups=layer_groups,
                        num_adj_samps=num_adj_samps)

MS.sample_network()

# Run the fitting procedure

Using the sampled adjacency matrix above, this is an example of how one can run the VB procedure.

In [4]:
VB = VariationalBayes(num_nodes=num_nodes, num_layers=num_layers,
                      adj_tensor=MS.adjacency_tensor, features=features,
                      M_w=3, M_z=3, num_fp_its=2, num_adj_samps=num_adj_samps, 
                      uniform_w=True)
VB.run_VB_scheme(n_CAVI_its=10, num_mc=10000, max_num_grad_steps=30, alpha=0.1,
                 alpha_set_theta=[0.1], alpha_set_sigma=[0.01],
                 eps_ELBO_phi=10**-8, max_ELBO_phi_dec=10, max_ELBO_dec=10,
                 ADAM_type='single', lr_theta=0.5, lr_sigma=0.1, verbose=1)

...Iteration 1 of 10...
...Updating rho...
...Updating gamma...
...Updating phi_0...
...Updating phi...
...Updating sigma2...30...
...Updating z...
...Updating w...
...Iteration 2 of 10...
...Updating rho...
...Updating gamma...
...Updating phi_0...
...Updating phi...
...Updating sigma2...10...
...Updating z...
...Updating w...
...Iteration 3 of 10...
...Updating rho...
...Updating gamma...
...Updating phi_0...
...Updating phi...
...Updating sigma2...10...
...Updating z...
...Updating w...
...Iteration 4 of 10...
...Updating rho...
...Updating gamma...
...Updating phi_0...
...Updating phi...
...Updating sigma2...10...
...Updating z...
...Updating w...
...Iteration 5 of 10...
...Updating rho...
...Updating gamma...
...Updating phi_0...
...Updating phi...
...Updating sigma2...10...
...Updating z...
...Updating w...
...Iteration 6 of 10...
...Updating rho...
...Updating gamma...
...Updating phi_0...
...Updating phi...
...Updating sigma2...10...
...Updating z...
...Updating w...
...Iterati

In [ ]:
global_groups = VB.phi_w_best.argmax(axis=1)
layer_groups = VB.phi_z.argmax(axis=2)

1.0
